# CP3 · Notebook 06 — Análisis de fallos

Categorizar tipos de error del VLM sobre las 18 imágenes. **Lo central del CP**.

~8 min.

In [ ]:
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

DATA = Path('../datasets/dashcam-curated')
OUT = Path('../outputs')

with open(OUT / '03_descriptions.json') as f:
    descriptions = json.load(f)
with open(OUT / '04_structured.json') as f:
    structured = json.load(f)

by_name_desc = {r['name']: r for r in descriptions}
by_name_struct = {r['name']: r for r in structured}

print(f'{len(descriptions)} descripciones cargadas')
print(f'{len(structured)} structured cargadas')

## 1. Taxonomía de fallos a mano

Vas a revisar **cada imagen** y categorizarla. **No automatizable** — esa es la lección.

In [ ]:
FAILURE_TYPES = {
    'correct':          'Descripción acertada, sin omisiones críticas.',
    'omission':         'No menciona un objeto/situación crítico visible.',
    'hallucination':    'Menciona objetos/elementos que NO están en la imagen.',
    'localization':    'Identifica objetos pero los ubica mal (lado, distancia).',
    'semantic_error':   'Identifica mal una categoría (e.g. confunde ciclista con peatón).',
    'ambiguous':        'Respuesta correcta pero no útil — demasiado genérica.',
}
for k, v in FAILURE_TYPES.items():
    print(f'  {k:18s} {v}')

## 2. Visualizar las 18 imágenes con su descripción

Repite para todas las categorías. Apunta en una libreta cuál encaja en cada `FAILURE_TYPE`.

In [ ]:
for r in descriptions:
    img = Image.open(DATA / r['path']).convert('RGB')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    ax1.imshow(img); ax1.set_title(f"{r['category']}: {r['name']}"); ax1.axis('off')
    desc_short = r['description'][:400] + ('...' if len(r['description']) > 400 else '')
    risk = by_name_struct.get(r['name'], {}).get('risk_score', '?')
    action = by_name_struct.get(r['name'], {}).get('action', '?')
    ax2.text(0.02, 0.96, f"Risk: {risk}  ·  Action: {action}\n\n{desc_short}",
             transform=ax2.transAxes, va='top', wrap=True, fontsize=9,
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#F5F2EE'))
    ax2.axis('off')
    plt.tight_layout(); plt.show()

## 3. Tu categorización a mano

Rellena este diccionario con tu juicio. **NO automatizable** — esa es la naturaleza del análisis crítico.

Para el entregable, **al menos 6 imágenes** debes categorizar (1 por categoría).

In [ ]:
# Plantilla — rellénala tras inspeccionar arriba
my_categorization = {
    # 'trivial_01.jpg':         'correct',
    # 'urbano_01.jpg':          'omission',
    # 'edge_visual_01.jpg':     'hallucination',
    # 'edge_semantic_01.jpg':   'semantic_error',
    # 'trampa_01.jpg':          'omission',
    # 'ambigua_01.jpg':         'ambiguous',
    # ... añade más
}

if not my_categorization:
    print('⚠️  Rellena `my_categorization` mirando los plots arriba')
else:
    from collections import Counter
    counts = Counter(my_categorization.values())
    print(f'{len(my_categorization)} imágenes categorizadas')
    for k, v in counts.most_common():
        print(f'  {k:18s} {v}')

## 4. Análisis: ¿qué tipo de fallo predomina?

Apunta para `07_local_vs_frontier.md`:

1. ¿En qué categoría sale **correct** más veces? (esperado: `trivial`, `urbano_standard`)
2. ¿Dónde aparece **hallucination**? (esperado: `edge_visual`, `trampa`)
3. ¿Dónde aparece **omission**? (esperado: `trampa`, `ambigua`)
4. ¿Hay un patrón claro?

**Pregunta clave**: si decidieras desplegar Moondream2 en producción, **dónde lo pondrías** (off-line auto-labeling vs on-line decision)? Argumenta con tus datos.

In [ ]:
with open(OUT / '06_categorization.json', 'w') as f:
    json.dump(my_categorization, f, indent=2)
print(f'✅ Tu categorización guardada en {OUT / "06_categorization.json"}')
print('Ve a 07_local_vs_frontier.md para el mini-ensayo del entregable.')